In [71]:
import pandas as pd

In [72]:
data = pd.read_csv("content/flipkart_product_review.csv")
data.head()

,product_id,product_title,rating,summary,review
0,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Terrific purchase,1-more flexible2-bass is very high3-sound clar...
1,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Terrific purchase,Super sound and good looking I like that prize
2,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Super!,Very much satisfied with the device at this pr...
3,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Super!,"Nice headphone, bass was very good and sound i..."
4,ACCFZGAQJGYCYDCM,BoAt Rockerz 235v2 with ASAP charging Version ...,5,Terrific purchase,Sound quality super battery backup super quali...


### **Convert data into Document format**

In [73]:
data = data[['product_title' , 'review']]
data.head()

,product_title,review
0,BoAt Rockerz 235v2 with ASAP charging Version ...,1-more flexible2-bass is very high3-sound clar...
1,BoAt Rockerz 235v2 with ASAP charging Version ...,Super sound and good looking I like that prize
2,BoAt Rockerz 235v2 with ASAP charging Version ...,Very much satisfied with the device at this pr...
3,BoAt Rockerz 235v2 with ASAP charging Version ...,"Nice headphone, bass was very good and sound i..."
4,BoAt Rockerz 235v2 with ASAP charging Version ...,Sound quality super battery backup super quali...


In [74]:
product_list = []

for index,row in data.iterrows():
    # Making a temp object
    object = {
        "product_name" : row["product_title"],
        "review" : row['review']
    }
    # Append the object to the product list
    product_list.append(object)

print(product_list[0])

{'product_name': 'BoAt Rockerz 235v2 with ASAP charging Version 5.0 Bluetooth Headset', 'review': "1-more flexible2-bass is very high3-sound clarity is good 4-battery back up to 6 to 8 hour's 5-main thing is fastest charging system is available in that. Only 20 min charge and get long up to 4 hours back up 6-killing look awesome 7-for gaming that product does not support 100% if you want for gaming then I'll recommend you please don't buy but you want for only music then this product is very well for you.. 8-no more wireless headphones are comparing with that headphones at this pric..."}


In [75]:
from langchain_core.documents import Document
docs = []

for object in product_list:
    metadata = {"product_name":object['product_name']}
    page_content = object['review']
    doc = Document(page_content=page_content , metadata = metadata)
    docs.append(doc)

In [76]:
docs[0]

Document(metadata={'product_name': 'BoAt Rockerz 235v2 with ASAP charging Version 5.0 Bluetooth Headset'}, page_content="1-more flexible2-bass is very high3-sound clarity is good 4-battery back up to 6 to 8 hour's 5-main thing is fastest charging system is available in that. Only 20 min charge and get long up to 4 hours back up 6-killing look awesome 7-for gaming that product does not support 100% if you want for gaming then I'll recommend you please don't buy but you want for only music then this product is very well for you.. 8-no more wireless headphones are comparing with that headphones at this pric...")

In [77]:
import os
from dotenv import load_dotenv
load_dotenv()

hugging_face_api = os.getenv("HF_TOKEN")

### **Store Embeddings to the Astra**

In [78]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7412.14it/s]


In [ ]:
astra_db_end_point = os.getenv("astra_db_end_point")
astra_db_app_token = os.getenv("astra_db_token")


AstraCS:rBzhHacSpDaKLrNANtBsYuDB:d755d10cdbc18c3ef8712dbc9e868854413a16c27e85ebde50aa314f3ac3aff8
https://3864b967-4e7e-423b-9475-11f0a73184f3-us-east-2.apps.astra.datastax.com


In [80]:
print("Uploading documents and generating embeddings... Please wait.")

from langchain_astradb import AstraDBVectorStore

vector_store = AstraDBVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    api_endpoint=astra_db_end_point,
    token=astra_db_app_token,
    collection_name="flipkart",
    namespace="default_keyspace"  # Maps to your keyspace name
)

Uploading documents and generating embeddings... Please wait.


### **LLM WORK**

In [97]:
from langchain_groq import ChatGroq
groq_api = os.getenv("groq_api")
model = ChatGroq(api_key=groq_api , model = "llama-3.3-70b-versatile")

In [98]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

# --- 1. Set up  Prompts & Retriever ---
retriever_prompt = (
    "Given a chat history and the latest user question which might reference context in the chat history, "
    "formulate a standalone question which can be understood without the chat history. "
    "Do NOT answer the question, just reformulate it if needed and otherwise return it as is."
)

contexulized_q_prompt = ChatPromptTemplate.from_messages([
    ('system' , retriever_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ('human' , '{input}')
])

PRODUCT_BOT_TEMPLATE = """
Your ecommercebot bot is an expert in product recommendations and customer queries.
It analyzes product titles and reviews to provide accurate and helpful responses.
Ensure your answers are relevant to the product context and refrain from straying off-topic.
Your responses should be concise and informative.

CONTEXT:
{context}

QUESTION: {input}

YOUR ANSWER:
"""

qa_prompt = ChatPromptTemplate.from_messages([
    ('system' , PRODUCT_BOT_TEMPLATE),
    MessagesPlaceholder(variable_name="chat_history"),
    ('human' , '{input}')
])


retriver = vector_store.as_retriever(search_kwargs = {'k' : 3})


# Custom History aware Retriver
condense_question_chain = contexulized_q_prompt | model | StrOutputParser()


def retrive_documents(inputs):
    # If there is chat history then condensed question chain first
    if inputs.get("chat_history"):
        standalone_question = condense_question_chain.invoke({
            "chat_history": inputs["chat_history"],
            "input": inputs["input"]
        })
    else:
        standalone_question = inputs['input']

    return retriver.invoke(standalone_question)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    # Step 1: Fetch and add the context to the input payload
    RunnablePassthrough.assign(
        context=RunnableLambda(retrive_documents) | format_docs
    )
    # Step 2: Pass everything to the prompt/model, and pack the result into an "answer" key
    | RunnablePassthrough.assign(
        answer=qa_prompt | model | StrOutputParser()  # <-- Wraps the string response into {"answer": "..."}
    )
)

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer", # Adjust based on whether your final model execution returns a string or object
)

In [99]:
chain_with_memory.invoke(
   {"input": "can you tell me the best bluetooth buds?"},
    config={
        "configurable": {"session_id": "Paras"}
    },  # constructs a key "abc123" in `store`.
)["answer"]

'Based on your experience, the Realme Buds Bluetooth seems to be a great option, with excellent sound quality, long battery life (up to 14 hours), and a smooth design. The metal filters inside the buds and fast Bluetooth connection also seem to be notable features. Would you like to know more about this product or compare it with other similar options?'

In [100]:
store 

{'Paras': InMemoryChatMessageHistory(messages=[HumanMessage(content='can you tell me the best bluetooth buds?', additional_kwargs={}, response_metadata={}), AIMessage(content='Based on your experience, the Realme Buds Bluetooth seems to be a great option, with excellent sound quality, long battery life (up to 14 hours), and a smooth design. The metal filters inside the buds and fast Bluetooth connection also seem to be notable features. Would you like to know more about this product or compare it with other similar options?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}

In [101]:
chain_with_memory.invoke(
   {"input": "what is my previous question?"},
    config={
        "configurable": {"session_id": "Paras"}
    },  # constructs a key "abc123" in `store`.
)["answer"]

'Your previous question was "can you tell me the best bluetooth buds?"'

In [102]:
store

{'Paras': InMemoryChatMessageHistory(messages=[HumanMessage(content='can you tell me the best bluetooth buds?', additional_kwargs={}, response_metadata={}), AIMessage(content='Based on your experience, the Realme Buds Bluetooth seems to be a great option, with excellent sound quality, long battery life (up to 14 hours), and a smooth design. The metal filters inside the buds and fast Bluetooth connection also seem to be notable features. Would you like to know more about this product or compare it with other similar options?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='what is my previous question?', additional_kwargs={}, response_metadata={}), AIMessage(content='Your previous question was "can you tell me the best bluetooth buds?"', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}